In [ ]:
# 連線 Kaggle 並安裝 DataBase 到 /cotent/data
# 選擇 Cancel upload

from google.colab import files
files.upload()  # 選 kaggle.json

!mkdir -p ~/.kaggle && echo KGAT_ac18305b6f0fc229afa6741b4b3a9c67 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
!pip -q install kaggle
!kaggle datasets download -d lakshaytyagi01/fruit-detection -p /content/data --unzip


Dataset URL: https://www.kaggle.com/datasets/lakshaytyagi01/fruit-detection
License(s): CC0-1.0
100% 501M/501M [00:03<00:00, 161MB/s]



In [ ]:
# 檢查 訓練集路徑 與 內容結構

import yaml
root = '/content/data/Fruits-detection'
p = f'{root}/data.yaml'
d = yaml.safe_load(open(p))
print(d)  # 先看原本內容，確認類別名稱

d['path'] = root
d['train'] = 'train/images'
d['val'] = 'valid/images'
d['test'] = 'test/images'
yaml.safe_dump(d, open(p, 'w'))

!ls /content/data/Fruits-detection/train


{'names': ['Apple', 'Banana', 'Grape', 'Orange', 'Pineapple', 'Watermelon'], 'nc': 6, 'path': '/content/data/Fruits-detection', 'test': 'test/images', 'train': 'train/images', 'val': 'valid/images'}
images	labels	labels.cache


In [ ]:
# pip install 必要套件

%pip -q install ultralytics kagglehub pyyaml

In [ ]:
# 開始訓練 (50輪, 解析度320x320, 每次看16張照片)

from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(data='/content/data/Fruits-detection/data.yaml', epochs=50, imgsz=320, batch=16)


# 訓練完後存檔

from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/runs /content/drive/MyDrive/fruit_runs_320

In [ ]:
# 在 Google Drive中的fruit_runs_320 中手動尋找訓練完成的.pt檔，接著複製到指定資料夾中 (/content/export320/best.pt)

from google.colab import drive
drive.mount('/content/drive')
model_path = '/content/drive/MyDrive/fruit_runs_320/detect/train/weights/best.pt'

!mkdir -p /content/export320/
!cp {model_path} /content/export320/best.pt


# 方便比較用
!mkdir -p /content/drive/MyDrive/export320
!cp {Model_path} /content/drive/MyDrive/export320/best.pt

In [ ]:
# ==========================================
# 專區 1：匯出並儲存 ONNX 模型
# ==========================================

from ultralytics import YOLO
import os

model_path = '/content/export320/best.pt'
model = YOLO(model_path)

# 執行 ONNX 格式匯出
print("開始匯出 ONNX 模型...")
onnx_path = model.export(format='onnx', imgsz=320, simplify=True)
print(f"ONNX 匯出成功，儲存於: {onnx_path}")

# 將生成的 onnx 檔案複製到 比較用的檔案夾中
!cp {onnx_path} /content/drive/MyDrive/export320
print("ONNX 模型已備份至 Google Drive！")

In [ ]:
# ==========================================
# 專區 2：匯出並儲存 NCNN 模型
# ==========================================

from ultralytics import YOLO
import os

# 確保 best.pt 存在（若前面已經掛載過 Google Drive 並複製過，可以直接用）
model_path = '/content/export320/best.pt'
if not os.path.exists(model_path):
    !mkdir -p /content/export320
    !cp /content/drive/MyDrive/fruit_runs_320/best.pt /content/export320/best.pt

model = YOLO(model_path)
print("目前模型類別：", model.names)  # 確認是 6 個水果類別

# 執行 NCNN 格式匯出
print("開始匯出 NCNN 模型...")
success_path = model.export(format='ncnn', imgsz=320)
print(f"NCNN 匯出成功，儲存於: {success_path}")

# 備份到 Google Drive
!mkdir -p /content/drive/MyDrive/fruit_export_320
# Ultralytics 導出 ncnn 時，資料夾通常會命名為 best_ncnn_model
!cp -r /content/export320/best_ncnn_model /content/drive/MyDrive/export320/ 2>/dev/null || cp -r best_ncnn_model /content/drive/MyDrive/export320/
print("NCNN 模型已備份至 Google Drive！")

In [ ]:
# 圖片測試
%pip -q install ultralytics

from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')

model = YOLO('/content/drive/MyDrive/export320/fruit_detect_320.onnx')

# 對測試集（或驗證集）進行評估並自動生成圖表
model.val(data='/content/data/Fruits-detection/data.yaml', split='test')

# model.predict('/content/data/Fruits-detection/test/images', save=True, conf=0.4)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/export320/fruit_detect_320.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.31.0 with CUDAExecutionProvider
Setting batch=1 input of shape (1, 3, 320, 320)



FileNotFoundError: Dataset '/content/data/Fruits-detection/data.yaml' images not found, missing path '/content/data/Fruits-detection/E:\DL\attempt1\attempt3\datasets\Fruits-detection-1\test\images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'